In [1]:
import torch

import cheetah

In [2]:
accelerated_device = "cuda"

In [3]:
incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=100_000,
    sigma_x=torch.tensor(1e-3),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)
histogram_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="histogram",
)
kde_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="kde",
)
cic_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="cloud-in-cell",
)

In [4]:
%%timeit
histogram_screen.track(incoming)
reading = histogram_screen.reading

torch.cpu.synchronize()

7.04 ms ± 127 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%%timeit
kde_screen.track(incoming)
reading = kde_screen.reading

torch.cpu.synchronize()

171 ms ± 4.99 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
%%timeit
cic_screen.track(incoming)
reading = cic_screen.reading

torch.cpu.synchronize()

10.8 ms ± 3.59 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
vectorized_incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=10_000,
    sigma_x=torch.linspace(1e-3, 2e-3, 100),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)

In [8]:
%%timeit
kde_screen.track(vectorized_incoming)
reading = kde_screen.reading

torch.cpu.synchronize()

781 ms ± 56.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit
cic_screen.track(vectorized_incoming)
reading = cic_screen.reading

torch.cpu.synchronize()

23.6 ms ± 1.63 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [10]:
tiny_incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=10,
    sigma_x=torch.tensor(1e-3),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)
tiny_histogram_screen = cheetah.Screen(
    resolution=(3, 4),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="histogram",
)
tiny_kde_screen = cheetah.Screen(
    resolution=(3, 4),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="kde",
)
tiny_cic_screen = cheetah.Screen(
    resolution=(3, 4),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="cloud-in-cell",
)

In [11]:
%%timeit
tiny_histogram_screen.track(tiny_incoming)
reading = tiny_histogram_screen.reading

torch.cpu.synchronize()

333 μs ± 1.42 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
%%timeit
tiny_kde_screen.track(tiny_incoming)
reading = tiny_kde_screen.reading

torch.cpu.synchronize()

932 μs ± 11.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [13]:
%%timeit
tiny_cic_screen.track(tiny_incoming)
reading = tiny_cic_screen.reading

torch.cpu.synchronize()

1.16 ms ± 104 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
mps_incoming = incoming.to(device=accelerated_device, dtype=torch.float32)
mps_histogram_screen = histogram_screen.to(device=accelerated_device, dtype=torch.float32)
mps_kde_screen = kde_screen.to(device=accelerated_device, dtype=torch.float32)
mps_cic_screen = cic_screen.to(device=accelerated_device, dtype=torch.float32)

In [15]:
# %%timeit
# mps_histogram_screen.track(mps_incoming)
# reading = mps_histogram_screen.reading
#
# getattr(torch, accelerated_device.type).synchronize()

In [16]:
%%timeit
mps_kde_screen.track(mps_incoming)
reading = mps_kde_screen.reading

getattr(torch, accelerated_device).synchronize()

9.47 ms ± 9.06 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [17]:
%%timeit
mps_cic_screen.track(mps_incoming)
reading = mps_cic_screen.reading

getattr(torch, accelerated_device).synchronize()

3.27 ms ± 19.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
mps_vectorized_incoming = vectorized_incoming.to(device=accelerated_device, dtype=torch.float32)

In [19]:
%%timeit
mps_kde_screen.track(mps_vectorized_incoming)
reading = mps_kde_screen.reading

getattr(torch, accelerated_device).synchronize()

72.2 ms ± 26.9 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [20]:
%%timeit
mps_cic_screen.track(mps_vectorized_incoming)
reading = mps_cic_screen.reading

getattr(torch, accelerated_device).synchronize()

3.86 ms ± 8.37 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
mps_tiny_incoming = tiny_incoming.to(device=accelerated_device, dtype=torch.float32)
mps_tiny_histogram_screen = tiny_histogram_screen.to(device=accelerated_device, dtype=torch.float32)
mps_tiny_kde_screen = tiny_kde_screen.to(device=accelerated_device, dtype=torch.float32)
mps_tiny_cic_screen = tiny_cic_screen.to(device=accelerated_device, dtype=torch.float32)

In [22]:
# %%timeit
# mps_tiny_histogram_screen.track(mps_tiny_incoming)
# reading = mps_tiny_histogram_screen.reading
#
# getattr(torch, accelerated_device).synchronize()

In [23]:
%%timeit
mps_tiny_kde_screen.track(mps_tiny_incoming)
reading = mps_tiny_kde_screen.reading

getattr(torch, accelerated_device).synchronize()

2.47 ms ± 6.92 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [24]:
%%timeit
mps_tiny_cic_screen.track(mps_tiny_incoming)
reading = mps_tiny_cic_screen.reading

getattr(torch, accelerated_device).synchronize()

3.19 ms ± 36 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
